# 🔍 Notebook 04 — Bias Analysis and Mitigation

This notebook runs all four **bias conditions** on the sentiment classifier and compares results:

| Condition | Parameter | Data | Loss |
|---|---|---|---|
| `probe` | Frozen base, head only | Original | CE |
| `lora` | LoRA r=8 | Original | CE |
| `lora_cda` | LoRA | + CDA (~30%) | CE |
| `lora_cda_rw` | LoRA | + CDA | Weighted CE (λ=2) |

Three bias probes are applied after each condition:
1. **Counterfactual** — gender/profession term swapping
2. **Positional** — text placement sensitivity
3. **Cultural** — ethnic/religious/regional group disparity

In [ ]:
import importlib, subprocess, sys
for pkg in ['torch','transformers','peft','datasets','accelerate','sentencepiece',
            'sacrebleu','bert-score','codecarbon','pyyaml','nvidia-ml-py']:
    if not importlib.util.find_spec(pkg.replace('-','_')):
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np, pandas as pd, torch

REPO = Path('.').resolve()
if REPO.name == 'notebooks': REPO = REPO.parent
sys.path.insert(0, str(REPO)); os.chdir(REPO)

from src.utils import load_config, set_all_seeds, MODEL_KEYS, ARCH_LABEL, run_dir, load_json
cfg = load_config()
SEED = cfg['experiment']['seeds'][0]
set_all_seeds(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
conditions = cfg['bias']['conditions']
print(f'Conditions: {conditions}')

## 1. Run All Conditions (skip if results exist)

In [ ]:
from src.training.sentiment import run_sentiment

for model_key in MODEL_KEYS:
    for cond in conditions:
        rd = run_dir(cfg['experiment']['results_dir'], 'sentiment', model_key, cond, SEED)
        if (rd / 'metrics.json').exists():
            print(f'  skip {model_key}/{cond} (exists)')
            continue
        print(f'\n=== {ARCH_LABEL[model_key]} | {cond} ===')
        try:
            run_sentiment(cfg, model_key, cond, SEED)
        except Exception as e:
            print(f'  FAILED: {e}')

## 2. Collect Bias Results

In [ ]:
def dig(d, path):
    for k in path.split('.'):
        d = d.get(k, {}) if isinstance(d, dict) else None
        if d is None: return None
    return d

rows = []
for m in MODEL_KEYS:
    for c in conditions:
        rd = run_dir(cfg['experiment']['results_dir'], 'sentiment', m, c, SEED)
        if not (rd / 'bias.json').exists(): continue
        b = load_json(rd / 'bias.json')
        met = load_json(rd / 'metrics.json')
        rows.append({'Architecture': ARCH_LABEL[m], 'Condition': c,
                     'Pos. flip': dig(b,'positional.flip_rate'),
                     'CF flip': dig(b,'counterfactual.flip_rate'),
                     'CF TV': dig(b,'counterfactual.tv'),
                     'Cultural disp.': dig(b,'cultural.disparity'),
                     'Robustness': b.get('bias_robustness'),
                     'Accuracy': met['accuracy'], 'Macro-F1': met['macro_f1']})
bias_df = pd.DataFrame(rows)
print(bias_df.to_markdown(index=False, floatfmt='.4f'))

## 3. Visualisation: Pre- vs Post-Mitigation

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

metrics = [('Pos. flip','Positional flip (↓)'),('CF flip','CF flip (↓)'),('Cultural disp.','Cultural disp. (↓)')]
if len(bias_df):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
    w = 0.8 / len(conditions)
    for ax, (col, title) in zip(axes, metrics):
        for j, c in enumerate(conditions):
            vals = [bias_df.loc[(bias_df['Architecture']==ARCH_LABEL[m])&(bias_df['Condition']==c), col].values
                    for m in MODEL_KEYS]
            vals = [v[0] if len(v) else np.nan for v in vals]
            ax.bar(np.arange(3)+j*w, vals, w, label=c)
        ax.set_xticks(np.arange(3)+w*(len(conditions)-1)/2, [ARCH_LABEL[m] for m in MODEL_KEYS], fontsize=9)
        ax.set_title(title)
    axes[0].legend(fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print('No bias results available yet. Run the cells above first.')

## 4. Cultural Bias Detail

In [ ]:
# Show per-group P(negative) for one model/condition
for m in MODEL_KEYS:
    rd = run_dir(cfg['experiment']['results_dir'], 'sentiment', m, 'lora', SEED)
    if not (rd / 'bias.json').exists(): continue
    b = load_json(rd / 'bias.json')
    records = dig(b, 'cultural.group_mean_p_negative') or []
    if records:
        cdf = pd.DataFrame(records)
        print(f'\n--- {ARCH_LABEL[m]} (lora): mean P(negative) per group ---')
        for cat in cdf['category'].unique():
            sub = cdf[cdf['category']==cat].sort_values('p_negative', ascending=False)
            print(f'  {cat}: {dict(zip(sub["group"], sub["p_negative"]))}')
    break

## 5. Key Takeaways

- **LoRA alone** reduces bias relative to the head-only probe in most cases.
- **CDA** provides additional reduction in counterfactual flip rate.
- **Loss reweighting** further improves robustness but may slightly decrease accuracy.
- All models exhibit measurable **cultural disparity** across ethnic/religious groups.
- See `docs/limitations.md` for caveats about the bias lexicon.

> ⚠️ The gender/profession swap lists have **not been validated by a native Amharic speaker**.